# 01 — Dados
Este caderno lê as 19 bases brutas da Economática e do BCB e as transforma em uma base mensal limpa.

In [1]:
from pathlib import Path
import pandas as pd

PASTA_RAW = Path("../data/raw")

arquivos = sorted(PASTA_RAW.glob("*"))
arquivos = [a for a in arquivos if a.name != ".gitkeep"]

print(len(arquivos), "arquivos em data/raw")
for a in arquivos:
    print(a.name)

19 arquivos em data/raw
010_EBITDA.xlsx
011_D&A.xlsx
012_FCFF.xlsx
013_Despesas_Financeiras.xlsx
01_cadastro_universo.xlsx
02_precos_fechamento_ajustados.xlsx
03_precos_sem_ajuste.xlsx
04_Volume_Diario.xlsx
05_Numeros_Negocios.xlsx
06_Valor_Mercado.xlsx
07_Quantidade_Acoes.xlsx
08_Patrimonio_Liquido.xlsx
09_divida_liquida.xlsx
14_ibov.xlsx
15_cdi_diario.csv
16_selic_meta.csv
17_selic_apurada.csv
18_di_1a_mensal.csv
19_ipca_mensal.csv


## 1. Como a Economática entrega os dados
Abrimos uma base sem nenhum tratamento, para ver o formato original.

In [2]:
caminho = PASTA_RAW / "09_divida_liquida.xlsx"
bruta = pd.read_excel(caminho, header=None, engine="openpyxl")

print("formato (linhas, colunas):", bruta.shape)
print()
print(bruta.iloc[:6, :3].to_string())

formato (linhas, colunas): (40, 1421)

                0                                                         1                                                                2
0  Nossa Caixa ON                                                       NaN                                                              NaN
1             NaN                                                       NaN                                                              NaN
2             NaN                                                       NaN                                                              NaN
3            Data   Div Tt Lq\n Em moeda orig\n em milhares\n consolid:sim*   Div Tt Lq\n Em moeda orig\n em milhares\n consolid:sim*\nTTEN3
4          1T2017                                                         -                                                                -
5          2T2017                                                         -                                        

In [3]:
nome_da_aba = pd.ExcelFile(caminho, engine="openpyxl").sheet_names[0].strip()
cabecalho = bruta.iloc[3].tolist()
codigos = [nome_da_aba] + [str(item).strip().split("\n")[-1].strip() for item in cabecalho[2:]]

print(nome_da_aba)
print(codigos[:5])
print(len(codigos))

BNCA3
['BNCA3', 'TTEN3', 'QVUM3B', 'QVQP3', 'APPA3']
1420


In [4]:
corpo = bruta.iloc[4:].reset_index(drop=True)
datas_brutas = corpo.iloc[:, 0]
valores = corpo.iloc[:, 1:].copy()
valores.columns = codigos
valores = valores.replace("-", float("nan")).apply(pd.to_numeric, errors="coerce")

print(datas_brutas.head(3).tolist(), "...", datas_brutas.tail(2).tolist())
print(valores.shape)
print("células com valor:", int(valores.notna().sum().sum()), "de", valores.size)

['1T2017', '2T2017', '3T2017'] ... ['3T2025', '4T2025']
(36, 1420)
células com valor: 22364 de 51120


/var/folders/3k/6sq134gx0w99q_v3_2rcjwh80000gn/T/ipykernel_67679/3963245790.py:5: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  valores = valores.replace("-", float("nan")).apply(pd.to_numeric, errors="coerce")


## 2. Datas
A Economática entrega os balanços como "1T2017". Convertemos para o último dia do trimestre (31/03/2017), que é a data a que o balanço se refere.

In [5]:
import re

def converter_trimestre(texto):
    trimestre, ano = re.fullmatch(r"([1-4])T(\d{4})", texto.strip()).groups()
    return pd.Period(year=int(ano), quarter=int(trimestre), freq="Q").end_time.normalize()

datas = [converter_trimestre(t) for t in datas_brutas]
valores.index = pd.DatetimeIndex(datas, name="data")

print(valores.index[:3].tolist())
print(valores.index[-2:].tolist())

[Timestamp('2017-03-31 00:00:00'), Timestamp('2017-06-30 00:00:00'), Timestamp('2017-09-30 00:00:00')]
[Timestamp('2025-09-30 00:00:00'), Timestamp('2025-12-31 00:00:00')]


## 3. Cadastro das ações
O cadastro diz, para cada código, se a ação está ativa ou cancelada, o setor e o segmento. Vamos conferir se ele cobre as ações da planilha.


In [6]:
cadastro = pd.read_excel(PASTA_RAW / "01_cadastro_universo.xlsx", header=None, engine="openpyxl", skiprows=4)
cadastro = cadastro.iloc[:, :14]
cadastro.columns = [
    "numero", "nome", "classe", "tipo_de_ativo", "bolsa", "situacao", "pais_sede",
    "nome_repetido", "codigo", "isin", "data_inicio_precos", "subsetor", "setor_economico", "segmento",
]

indice_codigos = pd.Index(codigos)
print("códigos na planilha:", len(indice_codigos), "| únicos:", indice_codigos.nunique())
print("repetidos:", indice_codigos[indice_codigos.duplicated()].tolist())
print("ações no cadastro:", cadastro["codigo"].nunique())
print("na planilha e fora do cadastro:", len(set(indice_codigos) - set(cadastro["codigo"].astype(str).str.strip())))
print(cadastro["situacao"].value_counts().to_dict())

códigos na planilha: 1420 | únicos: 1419
repetidos: ['BNCA3']
ações no cadastro: 1419
na planilha e fora do cadastro: 0
{'cancelado': 944, 'ativo': 475}


## 4. Lendo todas as bases
Juntamos os passos das seções 1 e 2 em uma função e a aplicamos às 12 bases em formato de matriz (data nas linhas, ação nas colunas).

In [7]:
MESES = {"Jan": 1, "Fev": 2, "Mar": 3, "Abr": 4, "Mai": 5, "Jun": 6,
         "Jul": 7, "Ago": 8, "Set": 9, "Out": 10, "Nov": 11, "Dez": 12}

def converter_data(valor):
    if hasattr(valor, "year"):
        return pd.Timestamp(valor)
    texto = str(valor).strip()
    if "T" in texto:
        trimestre, ano = texto.split("T")
        return pd.Timestamp(int(ano), int(trimestre) * 3, 1) + pd.offsets.MonthEnd(0)
    mes, ano = texto.split("-")
    return pd.Timestamp(int(ano), MESES[mes], 1) + pd.offsets.MonthEnd(0)

In [8]:
def ler_matriz(caminho):
    bruta = pd.read_excel(caminho, header=None, engine="openpyxl")
    aba = pd.ExcelFile(caminho, engine="openpyxl").sheet_names[0].strip()

    codigos = [aba] + [str(c).strip().split("\n")[-1].strip() for c in bruta.iloc[3, 2:]]
    datas = [converter_data(d) for d in bruta.iloc[4:, 0]]

    valores = bruta.iloc[4:, 1:].copy()
    valores.columns = codigos
    valores.index = pd.DatetimeIndex(datas, name="data")
    valores = valores.replace("-", float("nan")).apply(pd.to_numeric, errors="coerce")

    return valores.loc[:, ~valores.columns.duplicated()]

In [9]:
ARQUIVOS = {
    "preco_ajustado": "02_precos_fechamento_ajustados.xlsx",
    "preco_sem_ajuste": "03_precos_sem_ajuste.xlsx",
    "volume": "04_Volume_Diario.xlsx",
    "negocios": "05_Numeros_Negocios.xlsx",
    "valor_mercado": "06_Valor_Mercado.xlsx",
    "acoes": "07_Quantidade_Acoes.xlsx",
    "patrimonio_liquido": "08_Patrimonio_Liquido.xlsx",
    "divida_liquida": "09_divida_liquida.xlsx",
    "ebitda": "010_EBITDA.xlsx",
    "depreciacao": "011_D&A.xlsx",
    "fluxo_caixa_livre": "012_FCFF.xlsx",
    "despesa_financeira": "013_Despesas_Financeiras.xlsx",
}

matrizes = {}
for nome, arquivo in ARQUIVOS.items():
    matrizes[nome] = ler_matriz(PASTA_RAW / arquivo)
    m = matrizes[nome]
    print(f"{nome:20s} {m.shape}  {m.index[0].date()} a {m.index[-1].date()}")

/var/folders/3k/6sq134gx0w99q_v3_2rcjwh80000gn/T/ipykernel_67679/2211869328.py:11: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  valores = valores.replace("-", float("nan")).apply(pd.to_numeric, errors="coerce")


preco_ajustado       (2284, 1419)  2017-03-31 a 2025-12-31


/var/folders/3k/6sq134gx0w99q_v3_2rcjwh80000gn/T/ipykernel_67679/2211869328.py:11: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  valores = valores.replace("-", float("nan")).apply(pd.to_numeric, errors="coerce")


preco_sem_ajuste     (2284, 1419)  2017-03-31 a 2025-12-31


/var/folders/3k/6sq134gx0w99q_v3_2rcjwh80000gn/T/ipykernel_67679/2211869328.py:11: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  valores = valores.replace("-", float("nan")).apply(pd.to_numeric, errors="coerce")


volume               (2284, 1419)  2017-03-31 a 2025-12-31


... (13 outputs left out)

depreciacao          (36, 1419)  2017-03-31 a 2025-12-31


/var/folders/3k/6sq134gx0w99q_v3_2rcjwh80000gn/T/ipykernel_67679/2211869328.py:11: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  valores = valores.replace("-", float("nan")).apply(pd.to_numeric, errors="coerce")


fluxo_caixa_livre    (36, 1419)  2017-03-31 a 2025-12-31
despesa_financeira   (36, 1419)  2017-03-31 a 2025-12-31


/var/folders/3k/6sq134gx0w99q_v3_2rcjwh80000gn/T/ipykernel_67679/2211869328.py:11: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  valores = valores.replace("-", float("nan")).apply(pd.to_numeric, errors="coerce")


## 5. Ibovespa e taxa livre de risco

O Ibovespa entra só como benchmark. O CDI e a Selic servem de taxa livre de risco, para o fator de mercado e para o Sharpe. Primeiro olhamos o formato de cada arquivo, depois transformamos em séries mensais.

In [10]:
bruto = pd.read_excel(PASTA_RAW / "14_ibov.xlsx", header=None)

print(bruto.shape)
print(bruto.head(10))
print(bruto.tail(5))

(2353, 2)
                     0                                         1
0             Ibovespa                               Economatica
1                  NaN                                       NaN
2                  NaN                                       NaN
3                 Data  Fechamento\najust p/ prov\nEm moeda orig
4  2016-12-30 00:00:00                                         -
5  2017-01-02 00:00:00                                     59588
6  2017-01-03 00:00:00                                     61813
7  2017-01-04 00:00:00                                     61589
8  2017-01-05 00:00:00                                     62070
9  2017-01-06 00:00:00                                     61665
                        0          1
2348  2025-12-25 00:00:00          -
2349  2025-12-26 00:00:00  160896.64
2350  2025-12-29 00:00:00   160490.3
2351  2025-12-30 00:00:00  161125.37
2352  2025-12-31 00:00:00          -


In [11]:
ibov = bruto.iloc[4:].copy()
ibov.columns = ["data", "fechamento"]
ibov["data"] = pd.to_datetime(ibov["data"])
ibov["fechamento"] = pd.to_numeric(ibov["fechamento"], errors="coerce")
ibov = ibov.set_index("data")["fechamento"]

print(ibov.shape)
print(ibov.isna().sum(), "dias sem valor")
print(ibov.dropna().index.min().date(), "a", ibov.dropna().index.max().date())

(2349,)
115 dias sem valor
2017-01-02 a 2025-12-30


In [12]:
ibov_mensal = ibov.dropna().resample("ME").last()
ibov_ret = ibov_mensal.pct_change()

print(ibov_mensal.head(3))
print(ibov_ret.head(3))
print(ibov_mensal.shape)

data
2017-01-31    64670.0
2017-02-28    66662.0
2017-03-31    64984.0
Freq: ME, Name: fechamento, dtype: float64
data
2017-01-31         NaN
2017-02-28    0.030803
2017-03-31   -0.025172
Freq: ME, Name: fechamento, dtype: float64
(108,)


In [13]:
with open(PASTA_RAW / "15_cdi_diario.csv", encoding="utf-8") as f:
    linhas = [f.readline() for _ in range(6)]

for l in linhas:
    print(repr(l))

'"data";"valor"\n'
'"01/03/2017";"0,045442"\n'
'"02/03/2017";"0,045442"\n'
'"03/03/2017";"0,045442"\n'
'"06/03/2017";"0,045442"\n'
'"07/03/2017";"0,045442"\n'


In [14]:
cdi = pd.read_csv(PASTA_RAW / "15_cdi_diario.csv", sep=";", decimal=",")
cdi["data"] = pd.to_datetime(cdi["data"], format="%d/%m/%Y")
cdi = cdi.set_index("data")["valor"]

print(cdi.shape)
print(cdi.dtype)
print(cdi.isna().sum(), "sem valor")
print(cdi.index.min().date(), "a", cdi.index.max().date())

(2219,)
float64
0 sem valor
2017-03-01 a 2025-12-31


In [15]:
cdi_mensal = (1 + cdi / 100).resample("ME").prod() - 1

print(cdi_mensal.head(3))
print(cdi_mensal.tail(3))
print(cdi_mensal.shape)

data
2017-03-31    0.010504
2017-04-30    0.007853
2017-05-31    0.009256
Freq: ME, Name: valor, dtype: float64
data
2025-10-31    0.012757
2025-11-30    0.010527
2025-12-31    0.012199
Freq: ME, Name: valor, dtype: float64
(106,)


In [16]:
for nome in ["16_selic_meta.csv", "17_selic_apurada.csv"]:
    with open(PASTA_RAW / nome, encoding="utf-8") as f:
        linhas = [f.readline() for _ in range(5)]
    print(nome)
    for l in linhas:
        print(repr(l))
    print()

16_selic_meta.csv
'Data;432 - Taxa de juros - Meta Selic definida pelo Copom - % a.a.\n'
'31/03/2017;12,25\n'
'01/04/2017;12,25\n'
'02/04/2017;12,25\n'
'03/04/2017;12,25\n'

17_selic_apurada.csv
'\ufeffTaxa Selic - Dados diários;Filtros aplicados: Data inicial: 31/03/2017 / Data final: 31/12/2025.;;;;;;;;;\n'
'Data;Taxa (% a.a.);Fator diário;Financeiro (R$);Operações;Taxa média;Taxa mediana;Taxa modal;Desvio padrão;Índice de curtose;\n'
'31/12/2025;14,9;1,00055131;771.835.945.156,58;559;14,9;14,89;14,9;0,0265;598,5537;\n'
'30/12/2025;14,9;1,00055131;841.094.363.297,97;565;14,9;14,9;14,9;0,0262;896,1844;\n'
'29/12/2025;14,9;1,00055131;888.750.238.313,07;914;14,9;14,89;14,9;0,0288;669,0539;\n'



In [17]:
selic_meta = pd.read_csv(PASTA_RAW / "16_selic_meta.csv", sep=";", decimal=",", skiprows=1, skipfooter=1, engine="python", names=["data", "meta"])
selic_meta["data"] = pd.to_datetime(selic_meta["data"], format="%d/%m/%Y")
selic_meta = selic_meta.set_index("data")["meta"]

print(selic_meta.shape)
print(selic_meta.dtype)
print(selic_meta.isna().sum(), "sem valor")
print(selic_meta.index.min().date(), "a", selic_meta.index.max().date())
print(selic_meta[selic_meta.diff() != 0].to_string())

(3198,)
float64
0 sem valor
2017-03-31 a 2025-12-31
data
2017-03-31    12.25
2017-04-13    11.25
2017-06-01    10.25
2017-07-27     9.25
2017-09-08     8.25
2017-10-26     7.50
2017-12-07     7.00
2018-02-08     6.75
2018-03-22     6.50
2019-08-01     6.00
2019-09-19     5.50
2019-10-31     5.00
2019-12-12     4.50
2020-02-06     4.25
2020-03-19     3.75
2020-05-07     3.00
2020-06-18     2.25
2020-08-06     2.00
2021-03-18     2.75
2021-05-06     3.50
2021-06-17     4.25
2021-08-05     5.25
2021-09-23     6.25
2021-10-28     7.75
2021-12-09     9.25
2022-02-03    10.75
2022-03-17    11.75
2022-05-05    12.75
2022-06-16    13.25
2022-08-04    13.75
2023-08-03    13.25
2023-09-21    12.75
2023-11-02    12.25
2023-12-14    11.75
2024-02-01    11.25
2024-03-21    10.75
2024-05-09    10.50
2024-09-19    10.75
2024-11-07    11.25
2024-12-12    12.25
2025-01-30    13.25
2025-03-20    14.25
2025-05-08    14.75
2025-06-19    15.00


In [18]:
apurada = pd.read_csv(PASTA_RAW / "17_selic_apurada.csv", sep=";", decimal=",", skiprows=1, encoding="utf-8-sig", usecols=["Data", "Fator diário"])
apurada["Data"] = pd.to_datetime(apurada["Data"], format="%d/%m/%Y")
fator = apurada.set_index("Data")["Fator diário"].sort_index()

apurada_mensal = fator.resample("ME").prod() - 1
comparacao = pd.concat([cdi_mensal, apurada_mensal], axis=1, keys=["cdi", "apurada"]).loc["2017-04-30":]

print(fator.shape)
print(fator.index.min().date(), "a", fator.index.max().date())
print((comparacao["cdi"] - comparacao["apurada"]).abs().max())

(2197,)
2017-03-31 a 2025-12-31
1.5758088906858703e-05


## 6. Painel mensal por ação

Até aqui lemos os dados como vieram, em frequência diária, mensal e trimestral. O backtest trabalha em base mensal, então o primeiro passo é transformar os preços diários em fechamento e retorno mensal por ação. O fechamento é o último preço disponível de cada ação no mês, e o retorno é o fechamento dividido pelo do mês anterior, menos 1.

In [19]:
precos = matrizes["preco_ajustado"]

preco_mensal = precos.resample("ME").last()
retorno_mensal = preco_mensal / preco_mensal.shift(1) - 1

print(preco_mensal.shape)
print(preco_mensal.index.min().date(), "a", preco_mensal.index.max().date())
print(preco_mensal.notna().sum(axis=1).head(3))
print(preco_mensal.notna().sum(axis=1).tail(3))

(106, 1419)
2017-03-31 a 2025-12-31
data
2017-03-31    302
2017-04-30    389
2017-05-31    399
Freq: ME, dtype: int64
data
2025-10-31    410
2025-11-30    414
2025-12-31    412
Freq: ME, dtype: int64


In [20]:
r = retorno_mensal.stack()

print(r.describe())
print(r.nsmallest(5))
print(r.nlargest(5))

count    43333.000000
mean         0.020675
std          0.754762
min         -0.958920
25%         -0.062937
50%          0.000000
75%          0.070761
max        127.997345
dtype: float64
data              
2025-10-31  AMBP3    -0.958920
2025-07-31  GOLL54   -0.953873
2025-10-31  PDGR3    -0.928571
2024-08-31  AMER3    -0.895091
2024-09-30  PDGR3    -0.875000
dtype: float64
data             
2019-03-31  GSHP3    127.997345
2019-08-31  MAPT4     73.500000
2025-05-31  CCTY3     33.142857
2020-06-30  TXRX3     14.989000
2019-09-30  GPAR3     11.019231
dtype: float64


In [21]:
print((r == 0).mean())
print((r > 1).sum(), "retornos acima de +100%")

janela = slice("2019-02-20", "2019-04-05")
exame = pd.concat(
    [matrizes[k]["GSHP3"].loc[janela] for k in ["preco_ajustado", "preco_sem_ajuste", "volume"]],
    axis=1, keys=["ajustado", "sem_ajuste", "volume"],
).dropna(how="all")
print(exame.to_string())

print(matrizes["preco_ajustado"]["PDGR3"].loc["2025-09-25":"2025-10-31"].dropna().to_string())

0.016892437634135647
134 retornos acima de +100%
             ajustado  sem_ajuste    volume
data                                       
2019-02-20   0.276538        5.50    93.129
2019-02-21   0.271510        5.40    66.786
2019-02-22   0.296650        5.90  1152.443
2019-02-25   0.301678        6.00   912.474
2019-02-26   0.301678        6.00   732.133
2019-02-27   0.304192        6.05   117.194
2019-02-28   0.304192        6.05   285.126
2019-03-01   0.306203        6.09   170.894
2019-03-06   0.311231        6.19   116.313
2019-03-07   0.309723        6.16   181.199
2019-03-08   0.321790        6.40   149.060
2019-03-11   0.326818        6.50   559.388
2019-03-12   0.332852        6.62   466.976
2019-03-13   0.328829        6.54   224.478
2019-03-14   0.330338        6.57   192.702
2019-03-15   0.330338        6.57   206.178
2019-03-18   0.317768        6.32   410.400
2019-03-19   0.316762        6.30   144.114
2019-03-20   0.329332        6.55   277.333
2019-03-21   0.327321      

In [22]:
sem_ajuste = matrizes["preco_sem_ajuste"].resample("ME").last()
r_sem = sem_ajuste / sem_ajuste.shift(1) - 1

extremos = r[r > 1].rename("ajustado").to_frame()
extremos["sem_ajuste"] = [r_sem.loc[d, c] for d, c in extremos.index]

print(extremos.sort_values("ajustado", ascending=False).head(15).to_string())
print((extremos["sem_ajuste"] < 0).sum(), "casos com ajustado acima de +100% e sem ajuste negativo")

pdgr = pd.concat(
    [matrizes[k]["PDGR3"].loc["2025-09-30":"2025-10-15"] for k in ["preco_sem_ajuste", "volume"]],
    axis=1, keys=["sem_ajuste", "volume"],
).dropna(how="all")
print(pdgr.to_string())

                     ajustado  sem_ajuste
data                                     
2019-03-31 GSHP3   127.997345   -0.819835
2019-08-31 MAPT4    73.500000   73.500000
2025-05-31 CCTY3    33.142857   33.142857
2020-06-30 TXRX3    14.989000   14.989000
2019-09-30 GPAR3    11.019231   11.019231
2025-06-30 GOLL54    6.428852   -0.903401
2019-08-31 MEND5     6.202000    6.202000
2020-10-31 MMXM3     5.421348    5.421348
2019-08-31 TELB3     4.769231    4.769231
2020-06-30 TEKA3     4.720339    4.720339
2019-12-31 BRGE5     4.423221    4.423221
2024-03-31 TEKA3     4.417118    4.417118
2019-08-31 MEND6     4.356522    4.356522
2021-03-31 ASAI3     4.023267    4.023267
2019-12-31 EUCA3     3.733333    3.733333
9 casos com ajustado acima de +100% e sem ajuste negativo
            sem_ajuste    volume
data                            
2025-09-30        0.14   101.899
2025-10-01        0.13   133.504
2025-10-02        0.13   172.948
2025-10-03        0.07  2120.535
2025-10-06        0.04  1099.8

In [23]:
print(matrizes["volume"]["PETR4"].dropna().tail(5))
print(matrizes["preco_sem_ajuste"]["PETR4"].dropna().tail(3))

data
2025-12-22    1122827.287
2025-12-23    1083686.528
2025-12-26     610148.863
2025-12-29     632053.755
2025-12-30     520806.992
Name: PETR4, dtype: float64
data
2025-12-26    30.41
2025-12-29    30.73
2025-12-30    30.82
Name: PETR4, dtype: float64


Unidade do volume: o volume do PETR4 fica entre 520 mil e 1,1 milhão por dia, ou seja, R$ mil (cerca de R$ 0,5 a 1,1 bilhão por dia). O piso de liquidez usa R$ mil.

Se uma ação ficou mais de 10 dias corridos sem negociar antes do fim do mês, o preço do mês é descartado. Isso evita carregar preço velho de ação parada.

In [24]:
def fechamento_mensal(preco_diario, dias_maximos=10):
    ultima_data_com_preco = preco_diario.apply(lambda coluna: coluna.index.to_series().where(coluna.notna()))
    ultima_data_do_mes = ultima_data_com_preco.groupby(preco_diario.index.to_period("M")).max()
    fechamento = preco_diario.groupby(preco_diario.index.to_period("M")).last()
    fim_do_mes = fechamento.index.to_timestamp(how="end").normalize()
    atraso = ultima_data_do_mes.rsub(fim_do_mes, axis=0)
    dias_sem_negociar = atraso.apply(lambda coluna: coluna.dt.days)
    fechamento = fechamento.where(dias_sem_negociar <= dias_maximos)
    fechamento.index = fim_do_mes
    return fechamento

In [25]:
preco_ajustado_mensal = fechamento_mensal(matrizes["preco_ajustado"])
preco_sem_ajuste_mensal = fechamento_mensal(matrizes["preco_sem_ajuste"])
retorno_mensal_novo = preco_ajustado_mensal / preco_ajustado_mensal.shift(1) - 1

print(preco_ajustado_mensal.shape, preco_sem_ajuste_mensal.shape)
print("preços ajustados antes:", int(preco_mensal.notna().sum().sum()), "| depois:", int(preco_ajustado_mensal.notna().sum().sum()))
sem_ajuste_antigo = matrizes["preco_sem_ajuste"].resample("ME").last()
print("preços sem ajuste antes:", int(sem_ajuste_antigo.notna().sum().sum()), "| depois:", int(preco_sem_ajuste_mensal.notna().sum().sum()))
print("retornos válidos antes:", int(retorno_mensal.notna().sum().sum()), "| depois:", int(retorno_mensal_novo.notna().sum().sum()))
print(preco_ajustado_mensal.notna().sum(axis=1).head(3))
print(preco_ajustado_mensal.notna().sum(axis=1).tail(3))

(106, 1419) (106, 1419)
preços ajustados antes: 44978 | depois: 42911
preços sem ajuste antes: 44978 | depois: 42911
retornos válidos antes: 43333 | depois: 40756
data
2017-03-31    302
2017-04-30    351
2017-05-31    368
Freq: ME, dtype: int64
data
2025-10-31    395
2025-11-30    394
2025-12-31    389
Freq: ME, dtype: int64


A regra apagou 2.067 preços (44.978 para 42.911) e os retornos válidos caíram de 43.333 para 40.756.

In [26]:
apagados = preco_mensal.notna() & preco_ajustado_mensal.isna()
print(int(apagados.sum().sum()), "preços apagados")
print(apagados.sum().nlargest(8))

2067 preços apagados
SOND5    37
SOND6    37
CEEB5    34
TKNO4    30
EKTR3    29
JOPA4    27
BMIN3    27
USIM6    27
dtype: int64


## 7. Liquidez de 12 meses

Mediana, nos últimos 12 meses (mínimo 6), da mediana mensal do volume diário, ignorando dias sem negociação. Piso: R$ 1 milhão por dia (1000 em R$ mil).

In [27]:
LIQUIDEZ_MINIMA = 1000.0
DIA_SEM_NEGOCIACAO = "zeros"


def mediana_movel_12m(volume_diario):
    volume_por_mes = volume_diario.groupby(volume_diario.index.to_period("M")).median()
    mediana_movel = volume_por_mes.rolling(12, min_periods=6).median()
    mediana_movel.index = mediana_movel.index.to_timestamp(how="end").normalize()
    return mediana_movel


def volume_com_dias_zerados(volume_diario, preco_ajustado_diario):
    tem_preco = preco_ajustado_diario.notna()
    vida = tem_preco.cummax() & tem_preco.iloc[::-1].cummax().iloc[::-1]
    return volume_diario.fillna(0).where(vida)


liquidez_ignorando_dias = mediana_movel_12m(matrizes["volume"])
liquidez_com_zeros = mediana_movel_12m(volume_com_dias_zerados(matrizes["volume"], matrizes["preco_ajustado"]))
liquidez_12m = liquidez_com_zeros if DIA_SEM_NEGOCIACAO == "zeros" else liquidez_ignorando_dias

In [28]:
print(liquidez_12m.shape)
print(liquidez_12m["PETR4"].dropna().tail(3))
print(liquidez_12m["SOND5"].dropna().tail(3))
passa_no_piso = liquidez_12m >= LIQUIDEZ_MINIMA
print(passa_no_piso.sum(axis=1).iloc[[0, 5, 12, 40, 80, -1]])

(106, 1419)
data
2025-10-31    1.014097e+06
2025-11-30    1.014097e+06
2025-12-31    1.013253e+06
Freq: ME, Name: PETR4, dtype: float64
data
2025-10-31    0.0
2025-11-30    0.0
2025-12-31    0.0
Freq: ME, Name: SOND5, dtype: float64
data
2017-03-31      0
2017-08-31    138
2018-03-31    149
2020-07-31    184
2023-11-30    226
2025-12-31    204
dtype: int64


Dia sem negociação conta como zero dentro da vida da ação. O parâmetro DIA_SEM_NEGOCIACAO permite voltar à regra do legado ("ignorar").


## 8. Balanços alinhados ao mês

Um balanço só é público depois de um prazo: 45 dias após o fim do trimestre (ITR) e 90 dias após o 4T (DFP). Em cada fim de mês, a ação usa o último trimestre já publicado.

In [29]:
import numpy as np


def disponivel_em(fim_do_trimestre):
    prazo = np.where(fim_do_trimestre.month == 12, 90, 45)
    return fim_do_trimestre + pd.to_timedelta(prazo, unit="D")


def contabil_disponivel_em(tabela_trimestral, datas_de_decisao):
    datas_de_decisao = pd.DatetimeIndex(datas_de_decisao)
    divulgacao = disponivel_em(tabela_trimestral.index)
    posicao = np.searchsorted(divulgacao.to_numpy(), datas_de_decisao.to_numpy(), side="right") - 1
    valores = tabela_trimestral.to_numpy(dtype=float)[np.clip(posicao, 0, None)]
    valores[posicao < 0] = np.nan
    return pd.DataFrame(valores, index=datas_de_decisao, columns=tabela_trimestral.columns)


datas_mensais = preco_ajustado_mensal.index
contab_pl = contabil_disponivel_em(matrizes["patrimonio_liquido"], datas_mensais)

In [30]:
print(matrizes["patrimonio_liquido"]["PETR4"].loc["2017-03-31":"2018-06-30"])
print(contab_pl["PETR4"].loc["2017-03-31":"2018-06-30"])

data
2017-03-31    257820000
2017-06-30    258635000
2017-09-30    261764000
2017-12-31    263985000
2018-03-31    271963000
2018-06-30    282139000
Name: PETR4, dtype: int64
data
2017-03-31            NaN
2017-04-30            NaN
2017-05-31    257820000.0
2017-06-30    257820000.0
2017-07-31    257820000.0
2017-08-31    258635000.0
2017-09-30    258635000.0
2017-10-31    258635000.0
2017-11-30    261764000.0
2017-12-31    261764000.0
2018-01-31    261764000.0
2018-02-28    261764000.0
2018-03-31    263985000.0
2018-04-30    263985000.0
2018-05-31    271963000.0
2018-06-30    271963000.0
Freq: ME, Name: PETR4, dtype: float64


In [31]:
contab_pl = contabil_disponivel_em(matrizes["patrimonio_liquido"], datas_mensais)
contab_divida_liquida = contabil_disponivel_em(matrizes["divida_liquida"], datas_mensais)
contab_ebitda = contabil_disponivel_em(matrizes["ebitda"], datas_mensais)
contab_da = contabil_disponivel_em(matrizes["depreciacao"], datas_mensais)

for nome, tabela in [("pl", contab_pl), ("divida_liquida", contab_divida_liquida), ("ebitda", contab_ebitda), ("da", contab_da)]:
    print(nome, tabela.shape, int(tabela.notna().sum().sum()))

pl (106, 1419) 70447
divida_liquida (106, 1419) 64601
ebitda (106, 1419) 59158
da (106, 1419) 59158


In [32]:
tabela_teste = pd.DataFrame({"X": [1.0, 2.0, 3.0]}, index=pd.to_datetime(["2020-09-30", "2020-12-31", "2021-03-31"]))
datas_teste = pd.to_datetime(["2020-11-10", "2020-11-14", "2021-03-30", "2021-03-31", "2021-05-14", "2021-05-16"])

resultado = contabil_disponivel_em(tabela_teste, datas_teste)["X"].tolist()
assert np.allclose(resultado, [np.nan, 1, 1, 2, 2, 3], equal_nan=True), resultado

alterada = tabela_teste.copy()
alterada.loc["2021-03-31", "X"] = 99.0
antes = contabil_disponivel_em(tabela_teste, datas_teste)
depois = contabil_disponivel_em(alterada, datas_teste)
assert antes.loc[:"2021-05-14"].equals(depois.loc[:"2021-05-14"])
print("defasagem ok")

defasagem ok


In [33]:
alterada = matrizes["patrimonio_liquido"].copy()
alterada.loc["2017-12-31"] = alterada.loc["2017-12-31"] * 2
antes = contabil_disponivel_em(matrizes["patrimonio_liquido"], datas_mensais)
depois = contabil_disponivel_em(alterada, datas_mensais)
assert antes.loc[:"2018-02-28"].equals(depois.loc[:"2018-02-28"])
assert not antes.loc["2018-03-31"].equals(depois.loc["2018-03-31"])
print("defasagem na tabela real ok")

defasagem na tabela real ok


Em cada fim de mês, cada ação usa o último balanço já publicado (45 dias após o 1T a 3T, 90 dias após o 4T), sem preencher com o valor anterior. O teste confirma o prazo e que alterar um trimestre não muda nenhuma data anterior à publicação dele.


## 9. Cadastro: o que existe antes de filtrar

Antes de excluir ações, vemos os tipos de ativo, segmentos, setores e classes que o cadastro traz.

In [34]:
print(cadastro["tipo_de_ativo"].value_counts())
print(cadastro["segmento"].value_counts())
print(cadastro["setor_economico"].value_counts())
print(cadastro["classe"].value_counts())

tipo_de_ativo
Ação    1419
Name: count, dtype: int64
segmento
-                       632
Tradicional             350
Novo Mercado            230
Nível 2                  64
Nível 1                  62
Balcão Organizado        43
Bovespa Mais             18
Bovespa Mais Nível 2     13
BDR nível 3               6
BDR nível 2               1
Name: count, dtype: int64
setor_economico
-                                 583
Consumo cíclico                   187
Financeiro                        162
Bens industriais                  125
Utilidade pública                 119
Materiais básicos                  73
Consumo não cíclico                46
Saúde                              42
Outros                             25
Tecnologia da informação           22
Petróleo gás e biocombustíveis     19
Comunicações                       16
Name: count, dtype: int64
classe
ON         755
PN         411
PNA         82
PNB         69
UNT N2      22
PNC         16
UNT         15
PND          9
PNE    

In [35]:
SETORES_EXCLUIDOS = ["Financeiro"]
SEGMENTOS_EXCLUIDOS = ["Balcão Organizado", "Bovespa Mais", "Bovespa Mais Nível 2", "BDR nível 3", "BDR nível 2", "BDR nível 1"]


def codigos_permitidos(cadastro):
    permitido = ~cadastro["setor_economico"].isin(SETORES_EXCLUIDOS)
    permitido &= ~cadastro["segmento"].isin(SEGMENTOS_EXCLUIDOS)
    permitido &= ~cadastro["classe"].astype(str).str.contains("Resg")
    return set(cadastro.loc[permitido, "codigo"])


permitidos = codigos_permitidos(cadastro)

print("excluídos por setor:", int(cadastro["setor_economico"].isin(SETORES_EXCLUIDOS).sum()))
print("excluídos por segmento:", int(cadastro["segmento"].isin(SEGMENTOS_EXCLUIDOS).sum()))
print("excluídos por classe resgatável:", int(cadastro["classe"].astype(str).str.contains("Resg").sum()))
print("permitidos:", len(permitidos), "de", cadastro["codigo"].nunique())
print("units entre os permitidos:", int(cadastro.loc[cadastro["codigo"].isin(permitidos), "classe"].astype(str).str.contains("UNT|UnN1").sum()))

excluídos por setor: 162
excluídos por segmento: 81
excluídos por classe resgatável: 4
permitidos: 1191 de 1419
units entre os permitidos: 27


Todos os 1.419 ativos são do tipo "Ação"; o tipo não filtra. Saem o setor Financeiro (162), os segmentos fora de bolsa (81) e as classes resgatáveis (4), e sobram 1.191. Pendências: ativos sem setor (não dá para saber se são financeiros) e as 27 units que seguem no universo.

## 10. Uma classe por empresa

A empresa é identificada pela raiz do ISIN (caracteres 3 a 6); sem ISIN, pelas quatro primeiras letras do código. A classe que fica é a de maior liquidez entre as que passam nos outros filtros.

In [36]:
def raiz_da_empresa(cadastro):
    isin = cadastro["isin"].astype(str).str.strip()
    raiz = isin.str[2:6].where(isin.str.len() >= 6, cadastro["codigo"].astype(str).str[:4])
    return pd.Series(raiz.values, index=cadastro["codigo"].values)


sem_isin_valido = cadastro["isin"].astype(str).str.strip().str.len() < 6
raiz = raiz_da_empresa(cadastro)
raiz_permitidos = raiz[list(permitidos)]

print(raiz[["PETR3", "PETR4", "VALE3", "ABEV3"]])
print("sem ISIN válido:", int(sem_isin_valido.sum()), "| exemplos:", cadastro.loc[sem_isin_valido, "isin"].unique()[:3])
print("empresas entre os permitidos:", raiz_permitidos.nunique(), "| códigos:", len(raiz_permitidos))
print(raiz_permitidos.value_counts().head(5))

PETR3    PETR
PETR4    PETR
VALE3    VALE
ABEV3    ABEV
dtype: object
sem ISIN válido: 507 | exemplos: ['-']
empresas entre os permitidos: 752 | códigos: 1191
DROG    8
TMGC    7
MTDO    6
TMAC    6
CRBM    5
Name: count, dtype: int64


752 empresas entre os 1.191 códigos permitidos. Ativos sem ISIN (507, todos cancelados) são agrupados pelas quatro primeiras letras do código.

## 11. Universo elegível por mês

Elegível = permitida no cadastro, com preço no mês, preço sem ajuste de pelo menos R$ 1, liquidez no piso, valor de mercado positivo e, por empresa, só a classe de maior liquidez.

In [37]:
PRECO_MINIMO = 1.0


def elegivel_por_data(fechamento, preco_sem_ajuste_mensal, liquidez, valor_mercado, cadastro):
    colunas = [codigo for codigo in fechamento.columns if codigo in permitidos]
    raiz_colunas = raiz_da_empresa(cadastro)[colunas]
    passa = fechamento[colunas].notna()
    passa &= preco_sem_ajuste_mensal[colunas] >= PRECO_MINIMO
    passa &= liquidez[colunas] >= LIQUIDEZ_MINIMA
    passa &= valor_mercado[colunas] > 0
    liquidez_valida = liquidez[colunas].where(passa)
    elegivel = pd.DataFrame(False, index=fechamento.index, columns=fechamento.columns)
    for data in fechamento.index:
        linha = liquidez_valida.loc[data].dropna()
        if linha.empty:
            continue
        tabela = pd.DataFrame({"liquidez": linha, "raiz": raiz_colunas[linha.index]})
        escolhidos = tabela.sort_values("liquidez", ascending=False).drop_duplicates("raiz").index
        elegivel.loc[data, escolhidos] = True
    return elegivel


elegivel = elegivel_por_data(preco_ajustado_mensal, preco_sem_ajuste_mensal, liquidez_12m, matrizes["valor_mercado"], cadastro)
print(elegivel.shape)
print(elegivel.sum(axis=1).iloc[[0, 5, 12, 40, 80, -1]])
print("média por mês:", round(elegivel.sum(axis=1).iloc[5:].mean(), 1))

(106, 1419)
data
2017-03-31      0
2017-08-31    111
2018-03-31    115
2020-07-31    139
2023-11-30    171
2025-12-31    148
dtype: int64
média por mês: 153.8


In [38]:
financeiras = cadastro.loc[cadastro["setor_economico"] == "Financeiro", "codigo"]
print("financeiras elegíveis:", int(elegivel.reindex(columns=financeiras).sum().sum()))
print("PDGR3 em out/2025:", bool(elegivel.loc["2025-10-31", "PDGR3"]))
por_empresa = elegivel.T.groupby(raiz.reindex(elegivel.columns).values).sum().max().max()
print("máximo de classes por empresa em um mês:", int(por_empresa))

financeiras elegíveis: 0
PDGR3 em out/2025: False
máximo de classes por empresa em um mês: 1


Média de 153,8 elegíveis por mês (111 em ago/2017, 148 em dez/2025). Conferido: nenhuma financeira, no máximo uma classe por empresa e PDGR3 fora em out/2025.

## 12. Saídas e possíveis migrações

Lista as ações que param de ter preço antes do fim da base, com o retorno do último mês. Também procura códigos da mesma empresa que começam perto da saída (possível migração).

In [39]:
def tabela_de_saidas(fechamento, ultimo_mes_dos_dados):
    ultimo_mes_com_preco = pd.to_datetime(fechamento.apply(lambda coluna: coluna.last_valid_index()))
    saiu = ultimo_mes_com_preco < ultimo_mes_dos_dados
    retorno = fechamento.pct_change(fill_method=None)
    saidas = pd.DataFrame({"ultimo_mes_com_preco": ultimo_mes_com_preco[saiu]})
    saidas["retorno_no_ultimo_mes"] = [retorno.at[mes, codigo] for codigo, mes in saidas["ultimo_mes_com_preco"].items()]
    return saidas


saidas = tabela_de_saidas(preco_ajustado_mensal, preco_ajustado_mensal.index[-2])

print(len(saidas), "ações com último preço antes de", preco_ajustado_mensal.index[-2].date())
print(saidas["retorno_no_ultimo_mes"].describe())
print(saidas.reindex(["JBSS3", "JBSS32"]))

188 ações com último preço antes de 2025-11-30
count    149.000000
mean       0.010036
std        0.157332
min       -0.446889
25%       -0.067496
50%        0.007752
75%        0.076087
max        0.777143
Name: retorno_no_ultimo_mes, dtype: float64
       ultimo_mes_com_preco  retorno_no_ultimo_mes
JBSS3            2025-05-31              -0.067496
JBSS32                  NaT                    NaN


In [40]:
raiz = raiz_da_empresa(cadastro)
raiz = raiz[~raiz.index.duplicated()].reindex(preco_ajustado_mensal.columns)
quatro_letras = pd.Series([c[:4] for c in preco_ajustado_mensal.columns], index=preco_ajustado_mensal.columns)
primeiro_mes = pd.to_datetime(preco_ajustado_mensal.apply(lambda coluna: coluna.first_valid_index()))

candidatos = []
for codigo, mes_saida in saidas["ultimo_mes_com_preco"].items():
    mesma_empresa = (raiz == raiz[codigo]) | (quatro_letras == quatro_letras[codigo])
    comeca_perto = (primeiro_mes - mes_saida).abs() <= pd.Timedelta(days=62)
    for novo in preco_ajustado_mensal.columns[(mesma_empresa & comeca_perto).values]:
        if novo != codigo:
            candidatos.append({"saiu": codigo, "ultimo_mes": mes_saida.date(),
                               "novo": novo, "primeiro_mes_do_novo": primeiro_mes[novo].date()})

candidatos = pd.DataFrame(candidatos)
print(len(candidatos), "pares")
print(candidatos.head(40).to_string())

9 pares
     saiu  ultimo_mes    novo primeiro_mes_do_novo
0   AFLU3  2017-11-30   AFLU5           2017-11-30
1  BMGB11  2019-11-30   BMGB4           2019-11-30
2   CAMB4  2019-11-30   CAMB3           2019-12-31
3   JBSS3  2025-05-31  JBSS32           2025-06-30
4   LFFE4  2017-05-31   LFFE3           2017-03-31
5   MMAQ3  2024-12-31   MMAQ4           2024-12-31
6   MMAQ4  2024-12-31   MMAQ3           2024-12-31
7   SNSY6  2020-08-31   SNSY3           2020-08-31
8   SUZB5  2017-10-31   SUZB3           2017-11-30


188 ações saem da base antes de nov/2025 (149 com retorno no último mês). Dos 9 pares de códigos da mesma empresa perto de uma saída, alguns parecem conversões de classe (JBSS3→JBSS32, SUZB5→SUZB3, CAMB4→CAMB3); o resto é início da base ou série de um mês. Os pares confirmados recebem tratamento no mês da troca (seção seguinte).

## 13. Retornos extremos

Conta os retornos acima de +100% e abaixo de -50% entre as ações elegíveis. Não removemos nada por tamanho: só listamos para investigar caso a caso.

In [41]:
elegivel_no_mes_anterior = elegivel.shift(1, fill_value=False)
retorno_do_universo = retorno_mensal_novo.where(elegivel_no_mes_anterior)
r_universo = retorno_do_universo.stack().dropna()

print(len(r_universo), "retornos de ações elegíveis no mês anterior")
print((r_universo > 1).sum(), "acima de +100% |", (r_universo < -0.5).sum(), "abaixo de -50%")
print(r_universo.nlargest(10))
print(r_universo.nsmallest(10))

15346 retornos de ações elegíveis no mês anterior
10 acima de +100% | 40 abaixo de -50%
data              
2025-06-30  GOLL54    6.428852
2024-07-31  AMBP3     2.918605
2025-03-31  BHIA3     2.392453
2025-02-28  ONCO3     1.638498
2024-08-31  AERI3     1.265918
2024-11-30  AMER3     1.235135
2022-12-31  RCSL3     1.160763
2023-06-30  LIGT3     1.111111
2025-03-31  GFSA3     1.063636
2021-04-30  PMAM3     1.012009
dtype: float64
data              
2025-10-31  AMBP3    -0.958920
2023-01-31  AMER3    -0.818653
2025-12-31  AZUL54   -0.730462
2019-12-31  DMMO3    -0.718269
2024-01-31  GOLL54   -0.684504
2023-08-31  AALR3    -0.668323
2020-03-31  AMAR3    -0.664723
            MEAL3    -0.648718
2024-11-30  OIBR3    -0.643653
2020-03-31  SMLS3    -0.623246
dtype: float64


15.346 retornos de ações elegíveis: 10 acima de +100% e 40 abaixo de -50%. Vários parecem eventos reais (covid em mar/2020, Americanas em jan/2023); as altas em empresas em reestruturação precisam de conferência com o preço sem ajuste e a quantidade de ações.

In [42]:
retorno_sem_ajuste = preco_sem_ajuste_mensal / preco_sem_ajuste_mensal.shift(1) - 1
variacao_acoes = (
    matrizes["acoes"]
    .reindex(index=preco_ajustado_mensal.index, columns=preco_ajustado_mensal.columns)
    .pct_change(fill_method=None)
)

extremos = r_universo[(r_universo > 1) | (r_universo < -0.5)].rename("ajustado").to_frame()
extremos["sem_ajuste"] = [retorno_sem_ajuste.at[d, c] for d, c in extremos.index]
extremos["var_acoes"] = [variacao_acoes.at[d, c] for d, c in extremos.index]

print(extremos.sort_values("ajustado").round(3).to_string())

                   ajustado  sem_ajuste  var_acoes
data                                              
2025-10-31 AMBP3     -0.959      -0.959      0.000
2023-01-31 AMER3     -0.819      -0.819      0.000
2025-12-31 AZUL54    -0.730      -0.825      0.000
2019-12-31 DMMO3     -0.718      -0.718      0.000
2024-01-31 GOLL54    -0.685      -0.685      0.000
2023-08-31 AALR3     -0.668      -0.668      0.000
2020-03-31 AMAR3     -0.665      -0.665      0.000
           MEAL3     -0.649      -0.649      0.031
2024-11-30 OIBR3     -0.644      -0.644      4.552
2020-03-31 SMLS3     -0.623      -0.623      0.000
           BHIA3     -0.618      -0.618      0.000
           AZUL54    -0.605      -0.605      0.000
           COGN3     -0.604      -0.604      0.001
           RIAA3     -0.603      -0.603     -0.000
           PRIO3     -0.576      -0.576     -0.000
           EVEN3     -0.574      -0.574      0.004
           YDUQ3     -0.573      -0.573      0.000
           CVCB3     -0.569    

Dos 50 extremos, 46 têm retorno ajustado igual ao sem ajuste (eventos reais, como covid e Americanas). Dois têm sinal oposto, RCSL3 (dez/2022) e GOLL54 (jun/2025, ações x2.860), e são candidatos a erro de ajuste. Falta contar casos assim na base inteira antes de definir uma regra geral.

In [43]:
valor = matrizes["valor_mercado"].reindex(index=preco_ajustado_mensal.index, columns=preco_ajustado_mensal.columns)
var_valor_mercado = valor.pct_change(fill_method=None)

conferencia = pd.concat(
    {"ajustado": retorno_do_universo, "sem_ajuste": retorno_sem_ajuste,
     "var_acoes": variacao_acoes, "var_valor_mercado": var_valor_mercado},
    axis=1,
).stack(level=1).dropna(subset=["ajustado"])
conferencia["diferenca"] = conferencia["ajustado"] - conferencia["sem_ajuste"]

opostos = conferencia[(conferencia["ajustado"] * conferencia["sem_ajuste"] < 0) & (conferencia["diferenca"].abs() > 0.3)]

print(len(conferencia), "retornos conferidos")
print(len(opostos), "com sinal oposto e diferença acima de 30 pontos")
print(opostos.sort_values("diferenca").round(3).to_string())
print(conferencia[conferencia["ajustado"] > 1].round(3).to_string())

15346 retornos conferidos
42 com sinal oposto e diferença acima de 30 pontos
                   ajustado  sem_ajuste  var_acoes  var_valor_mercado  diferenca
data                                                                            
2025-06-30 GFSA3     -0.143      16.131     -0.950             -0.143    -16.275
2019-05-31 PMAM3     -0.132      13.764     -0.941             -0.132    -13.896
2025-06-30 HAPV3     -0.141      11.885     -0.933             -0.140    -12.026
2024-05-31 MGLU3     -0.089       8.110     -0.900             -0.089     -8.199
2022-09-30 GFSA3     -0.246       5.782     -0.889             -0.246     -6.028
2020-08-31 HBOR3     -0.196       3.020     -0.800             -0.196     -3.216
2024-05-31 RCSL3     -0.277       1.892     -0.750             -0.334     -2.169
2023-07-31 RCSL3     -0.404       0.193     -0.500             -0.182     -0.596
2022-06-30 RCSL3     -0.498       0.003     -0.500             -0.453     -0.502
2025-11-30 EGIE3      0.072     

/var/folders/3k/6sq134gx0w99q_v3_2rcjwh80000gn/T/ipykernel_67679/3882887891.py:8: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.
  ).stack(level=1).dropna(subset=["ajustado"])


37 dos 42 casos de sinal oposto são desdobramentos e grupamentos bem ajustados (o ajustado acompanha o valor de mercado). As outras 8 altas acima de +100% têm os três retornos iguais: movimento real. Cinco casos não batem com nenhuma medida: GOLL54 (jun/2025), RCSL3 (dez/2022), BRAP4 (dez/2021), JSLG3 (dez/2025) e AZEV4 (fev/2025).

In [44]:
LIMITE_DIVERGENCIA = 0.3

conferencia["dif_sem_ajuste"] = (conferencia["ajustado"] - conferencia["sem_ajuste"]).abs()
conferencia["dif_valor_mercado"] = (conferencia["ajustado"] - conferencia["var_valor_mercado"]).abs()

for limite in [0.2, 0.3, 0.5]:
    n = int(((conferencia["dif_sem_ajuste"] > limite) & (conferencia["dif_valor_mercado"] > limite)).sum())
    print("limite", limite, "->", n, "casos")

inexplicado = (conferencia["dif_sem_ajuste"] > LIMITE_DIVERGENCIA) & (conferencia["dif_valor_mercado"] > LIMITE_DIVERGENCIA)
print(conferencia[inexplicado].round(3).to_string())

limite 0.2 -> 15 casos
limite 0.3 -> 8 casos
limite 0.5 -> 3 casos
                   ajustado  sem_ajuste  var_acoes  var_valor_mercado  diferenca  dif_sem_ajuste  dif_valor_mercado
data                                                                                                               
2021-07-31 LAME4     -0.207      -0.671      0.000             -0.675      0.464           0.464              0.467
2021-12-31 BRAP4      0.218      -0.498      0.000             -0.512      0.716           0.716              0.730
2022-12-31 RCSL3      1.161      -0.101      0.000             -0.178      1.261           1.261              1.339
2023-08-31 PCAR3     -0.424      -0.761      0.000             -0.761      0.338           0.338              0.338
2024-09-30 CASH3     -0.045      -0.412      0.001             -0.411      0.367           0.367              0.366
2025-02-28 AZEV4      0.280      -0.100      2.000              1.691      0.380           0.380              1.411
2025-

Proposta, a decidir com o grupo: sinalizar como suspeito o retorno em que o ajustado diverge do sem ajuste e do valor de mercado por mais de 1,0. O limite foi escolhido depois de ver os dados; qualquer valor entre ~0,73 e ~1,26 dá os mesmos 2 casos (RCSL3 dez/2022 e GOLL54 jun/2025). Padrão: manter, com a lista guardada.

In [45]:
LIMITE_DIVERGENCIA = 1.0
AJUSTE_INEXPLICADO = "manter"

dif_sem_ajuste = (retorno_mensal_novo - retorno_sem_ajuste).abs()
dif_valor_mercado = (retorno_mensal_novo - var_valor_mercado).abs()
suspeitos = (dif_sem_ajuste > LIMITE_DIVERGENCIA) & (dif_valor_mercado > LIMITE_DIVERGENCIA)

casos = suspeitos.stack()
casos = casos[casos].index
print(len(casos), "retornos suspeitos na base inteira")
for data, codigo in casos:
    print(data.date(), codigo, round(retorno_mensal_novo.at[data, codigo], 3),
          round(retorno_sem_ajuste.at[data, codigo], 3), round(var_valor_mercado.at[data, codigo], 3))

if AJUSTE_INEXPLICADO == "anular":
    retorno_mensal_tratado = retorno_mensal_novo.mask(suspeitos)
else:
    retorno_mensal_tratado = retorno_mensal_novo.copy()

print(int(retorno_mensal_novo.notna().sum().sum()), "->", int(retorno_mensal_tratado.notna().sum().sum()), "retornos válidos")

10 retornos suspeitos na base inteira
2018-11-30 RCSL3 1.724 0.349 0.12
2019-03-31 GSHP3 127.997 -0.82 -0.82
2019-03-31 IDVL4 1.43 0.401 -0.234
2020-10-31 AZEV3 1.512 0.124 0.215
2022-12-31 RCSL3 1.161 -0.101 -0.178
2023-03-31 PDGR3 -0.573 41.7 1.462
2023-04-30 BRPR3 2.559 -0.729 -0.729
2024-12-31 FASA3 1.214 -0.323 -0.312
2025-06-30 GOLL54 6.429 -0.903 275.497
2025-08-31 RVEE3 2.348 -0.856 0.44
40756 -> 40756 retornos válidos


10 retornos suspeitos na base inteira (2 no universo elegível). "Suspeito" = o ajustado diverge do sem ajuste e do valor de mercado por mais de 1,0; ainda não é erro confirmado. Padrão "manter": nada foi alterado.

In [46]:
def olhar_caso(codigo, data):
    inicio = data.replace(day=1)
    tabela = pd.concat(
        [matrizes["preco_ajustado"][codigo], matrizes["preco_sem_ajuste"][codigo]],
        axis=1, keys=["ajustado", "sem_ajuste"],
    ).loc[inicio - pd.Timedelta(days=10): data].dropna(how="all")
    tabela["ret_ajustado"] = tabela["ajustado"].pct_change(fill_method=None)
    tabela["ret_sem_ajuste"] = tabela["sem_ajuste"].pct_change(fill_method=None)
    dia = tabela.loc[inicio:, "ret_ajustado"].abs().idxmax()
    return tabela.loc[dia - pd.Timedelta(days=5): dia + pd.Timedelta(days=5)].round(3)


for data, codigo in casos:
    print(codigo, data.date())
    print(olhar_caso(codigo, data).to_string())
    print()

RCSL3 2018-11-30
            ajustado  sem_ajuste  ret_ajustado  ret_sem_ajuste
data                                                          
2018-11-23    13.188        1.98         0.359          -0.327
2018-11-26    22.379        3.36         0.697           0.697
2018-11-27    35.966        5.40         0.607           0.607
2018-11-28    30.904        4.64        -0.141          -0.141
2018-11-29    28.706        4.31        -0.071          -0.071
2018-11-30    25.976        3.90        -0.095          -0.095

GSHP3 2019-03-31
            ajustado  sem_ajuste  ret_ajustado  ret_sem_ajuste
data                                                          
2019-03-22     0.332        6.60         0.014           0.014
2019-03-25     0.353        7.03         0.065           0.065
2019-03-26     0.360        7.16         0.018           0.018
2019-03-27    50.400        1.40       138.999          -0.804
2019-03-28    45.000        1.25        -0.107          -0.107
2019-03-29    39.240

Dos 10 suspeitos, 6 mostram o ajustado saltando num dia em que o preço cai (GSHP3, RCSL3 dez/2022, AZEV3, BRPR3, FASA3, GOLL54): provável erro de ajuste. RCSL3 nov/2018 e RVEE3 são ambíguos. IDVL4 e PDGR3 são falsos positivos da regra mensal (ajuste correto). Nada foi alterado; a lista vai ao grupo.

In [47]:
pares = [("JBSS3", "JBSS32"), ("VVAR3", "BHIA3"), ("LAME4", "AMER3"),
         ("SUZB5", "SUZB3"), ("CAMB4", "CAMB3"), ("AFLU3", "AFLU5"), ("BMGB11", "BMGB4")]
sem_ajuste_diario = matrizes["preco_sem_ajuste"]

linhas = []
for antigo, novo in pares:
    if antigo not in sem_ajuste_diario.columns or novo not in sem_ajuste_diario.columns:
        linhas.append({"antigo": antigo, "novo": novo})
        continue
    ultimo = sem_ajuste_diario[antigo].last_valid_index()
    primeiro = sem_ajuste_diario[novo].first_valid_index()
    p_antigo = sem_ajuste_diario[antigo].loc[ultimo]
    p_novo = sem_ajuste_diario[novo].loc[primeiro]
    linhas.append({"antigo": antigo, "novo": novo, "ultimo_dia_antigo": ultimo.date(),
                   "primeiro_dia_novo": primeiro.date(), "dias_entre": (primeiro - ultimo).days,
                   "preco_antigo": p_antigo, "preco_novo": p_novo, "razao": round(p_novo / p_antigo, 3)})

print(pd.DataFrame(linhas).to_string())

   antigo    novo ultimo_dia_antigo primeiro_dia_novo  dias_entre  preco_antigo  preco_novo  razao
0   JBSS3  JBSS32        2025-06-06        2025-06-09         3.0         39.03       78.20  2.004
1   VVAR3   BHIA3               NaN               NaN         NaN           NaN         NaN    NaN
2   LAME4   AMER3        2022-01-21        2017-03-31     -1757.0          6.55       12.35  1.885
3   SUZB5   SUZB3        2017-11-09        2017-11-10         1.0         21.13       20.56  0.973
4   CAMB4   CAMB3        2019-12-20        2019-12-23         3.0          6.25        6.50  1.040
5   AFLU3   AFLU5        2017-11-22        2017-11-16        -6.0          5.15        5.10  0.990
6  BMGB11   BMGB4        2019-11-28        2019-11-29         1.0         33.60        8.40  0.250


In [48]:
setor = (
    cadastro.assign(codigo=cadastro["codigo"].astype(str).str.strip())
    .drop_duplicates("codigo").set_index("codigo")["setor_economico"]
    .reindex(elegivel.columns)
)
sem_setor = (setor.isna() | setor.astype(str).str.strip().isin(["-", "", "nan"])).values
print(int(sem_setor.sum()), "ações sem setor no cadastro")
print("elegíveis sem setor por mês (média):", round(elegivel.loc[:, sem_setor].sum(axis=1).mean(), 1))
print(sorted(elegivel.columns[sem_setor & elegivel.any().values])[:40])

583 ações sem setor no cadastro
elegíveis sem setor por mês (média): 0.0
[]


In [49]:
print([c for c in matrizes["preco_sem_ajuste"].columns if c.startswith(("VVAR", "VIIA", "BHIA", "LAME", "AMER", "BTOW", "B2W"))])

['AMER3', 'BHIA3', 'VVAR4', 'VVAR11', 'LAME3', 'LAME4', 'LAME98', 'LAME99']


## 14. Mês da troca de código

Em cinco trocas confirmadas (JBSS3, SUZB5, CAMB4, AFLU3, LAME4), o retorno do código antigo no mês da troca é: retorno do antigo até o último pregão dele, vezes o retorno do novo desde o primeiro pregão seguinte. VVAR→BHIA3 não precisa (o BHIA3 já traz o histórico). Parâmetro TRATAR_MIGRACAO.

In [50]:
PARES_MIGRACAO = [("JBSS3", "JBSS32"), ("SUZB5", "SUZB3"), ("CAMB4", "CAMB3"),
                  ("AFLU3", "AFLU5"), ("LAME4", "AMER3")]
TRATAR_MIGRACAO = True


def retorno_do_mes_da_troca(antigo, novo, diario, mensal):
    serie_antiga = diario[antigo].dropna()
    ultimo_dia = serie_antiga.index[-1]
    ultimo_mes = mensal[antigo].last_valid_index()
    mes_da_troca = mensal.index[mensal.index.get_loc(ultimo_mes) + 1]
    primeiro_dia_novo = diario[novo].loc[ultimo_dia + pd.Timedelta(days=1):].first_valid_index()
    parte_antiga = serie_antiga.iloc[-1] / mensal.at[ultimo_mes, antigo]
    parte_nova = mensal.at[mes_da_troca, novo] / diario[novo].loc[primeiro_dia_novo]
    return mes_da_troca, parte_antiga * parte_nova - 1


retorno_mensal_final = retorno_mensal_novo.copy()
if TRATAR_MIGRACAO:
    for antigo, novo in PARES_MIGRACAO:
        mes, retorno = retorno_do_mes_da_troca(antigo, novo, matrizes["preco_ajustado"], preco_ajustado_mensal)
        print(antigo, "->", novo, mes.date(), "antes:", retorno_mensal_final.at[mes, antigo], "| depois:", round(retorno, 3))
        retorno_mensal_final.at[mes, antigo] = retorno

JBSS3 -> JBSS32 2025-06-30 antes: nan | depois: -0.027
SUZB5 -> SUZB3 2017-11-30 antes: nan | depois: -0.106
CAMB4 -> CAMB3 2019-12-31 antes: nan | depois: 0.094
AFLU3 -> AFLU5 2017-12-31 antes: nan | depois: nan
LAME4 -> AMER3 2022-02-28 antes: nan | depois: -0.088


Quatro das cinco trocas deram retorno plausível no mês da troca (JBSS3 -2,7%, SUZB5 -10,6%, CAMB4 +9,4%, LAME4 -8,8%). AFLU3 ficou vazio: o código novo não tem preço de fim de mês. Com TRATAR_MIGRACAO = False, nada muda.

## 15. Gravar os arquivos do 01

Grava em data/processed os arquivos do contrato (seção 26 do documento), que o 02, 03 e 04 leem. Os .pkl não vão para o Git: cada pessoa roda este caderno.

In [51]:
PASTA_SAIDA = Path("../data/processed")
PASTA_SAIDA.mkdir(parents=True, exist_ok=True)

colunas = preco_ajustado_mensal.columns
datas = preco_ajustado_mensal.index

cadastro_universo = (
    cadastro.assign(codigo=cadastro["codigo"].astype(str).str.strip(),
                    raiz_isin=raiz_da_empresa(cadastro).values)
    .drop_duplicates("codigo").set_index("codigo")
    [["isin", "classe", "tipo_de_ativo", "setor_economico", "subsetor", "segmento", "situacao", "raiz_isin"]]
    .reindex(colunas)
)

arquivos_saida = {
    "cadastro_universo": cadastro_universo,
    "retorno_mensal": retorno_mensal_final,
    "preco_sem_ajuste_mensal": preco_sem_ajuste_mensal,
    "valor_mercado_mensal": matrizes["valor_mercado"].reindex(index=datas, columns=colunas),
    "liquidez_12m": liquidez_12m,
    "contab_pl": contab_pl,
    "contab_divida_liquida": contab_divida_liquida,
    "contab_ebitda": contab_ebitda,
    "contab_da": contab_da,
    "elegivel": elegivel,
    "saidas": saidas,
    "cdi_mensal": cdi_mensal.reindex(datas),
    "ibov_mensal": ibov_ret.reindex(datas),
}

for nome, tabela in arquivos_saida.items():
    if isinstance(tabela, pd.DataFrame) and tabela.shape == (len(datas), len(colunas)):
        arquivos_saida[nome] = tabela.reindex(index=datas, columns=colunas)

for nome, objeto in arquivos_saida.items():
    pd.to_pickle(objeto, PASTA_SAIDA / f"{nome}.pkl")
    print(nome, objeto.shape)

cadastro_universo (1419, 8)
retorno_mensal (106, 1419)
preco_sem_ajuste_mensal (106, 1419)
valor_mercado_mensal (106, 1419)
liquidez_12m (106, 1419)
contab_pl (106, 1419)
contab_divida_liquida (106, 1419)
contab_ebitda (106, 1419)
contab_da (106, 1419)
elegivel (106, 1419)
saidas (188, 2)
cdi_mensal (106,)
ibov_mensal (106,)


## 16. DI de 1 ano e IPCA
Alinha o DI (B3) e o IPCA (IBGE) às 106 datas e grava em `data/processed`. O DI só entra no diagnóstico de juros; o IPCA, no retorno real opcional.

In [52]:
di = pd.read_csv(PASTA_RAW / '18_di_1a_mensal.csv', parse_dates=['mes', 'data_pregao']).set_index('mes')
ipca = pd.read_csv(PASTA_RAW / '19_ipca_mensal.csv', parse_dates=['mes']).set_index('mes')

assert di.index.equals(datas) and ipca.index.equals(datas)
atraso = (di.index - di['data_pregao']).dt.days
assert atraso.between(0, 5).all()
assert di['taxa_252'].notna().all() and ipca['ipca_decimal'].notna().all()

di_mensal = di[['taxa_252', 'delta_taxa']]
ipca_mensal = ipca['ipca_decimal']
pd.to_pickle(di_mensal, PASTA_SAIDA / 'di_mensal.pkl')
pd.to_pickle(ipca_mensal, PASTA_SAIDA / 'ipca_mensal.pkl')
print(di_mensal.shape, ipca_mensal.shape, int(atraso.max()), 'dias de atraso máximo do pregão')

(106, 2) (106,) 3 dias de atraso máximo do pregão
